In [ ]:
# =========================================================
# 基于摄像头的森林火情实时监测预警系统（华为云IoT融合版）
# 改造自 MediaPipe Tasks 原始项目，移除人体姿态，改用色彩+动态火焰识别
# 适用场景：林区监控、野外摄像头火情检测
# 识别逻辑：HSV火焰色彩阈值 + 帧间动态变化积分判定火焰
# 预警分级：轻微火情预警 / 严重火灾高危警报
# 华为云IoT：接收co2/smoke/alert_level，上报fire_type(long数字等级)
# =========================================================

import os
import sys
import ssl
import time
import math
import json
import threading
import urllib.request
import subprocess
import warnings
from collections import deque

warnings.filterwarnings("ignore")


# =========================================================
# 1. 自动依赖安装
# =========================================================
try:
    import cv2
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "opencv-python",
        "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"
    ])
    import cv2

try:
    import numpy as np
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "numpy",
        "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"
    ])
    import numpy as np

try:
    from PIL import Image, ImageDraw, ImageFont
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "pillow",
        "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"
    ])
    from PIL import Image, ImageDraw, ImageFont

try:
    from IPython.display import display as notebook_display
except Exception:
    notebook_display = None

try:
    from IPython.display import clear_output, display, HTML
except Exception:
    clear_output = None
    display = None
    HTML = None

# MQTT依赖
try:
    from paho.mqtt import client as mqtt_client
    from paho.mqtt.client import CallbackAPIVersion
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "paho-mqtt",
        "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"
    ])
    from paho.mqtt import client as mqtt_client
    from paho.mqtt.client import CallbackAPIVersion


# =========================================================
# 2. 全局路径与硬件参数配置
# =========================================================
PROJECT_DIR = r"D:\torch_env\AI_FireMonitor"
SAVE_DIR = PROJECT_DIR

CAMERA_INDEX = 0
FRAME_WIDTH = 960
FRAME_HEIGHT = 540
FPS_TARGET = 30

# HSV火焰颜色阈值（适配火光：橙红、亮黄、明火）
FIRE_LOW_HSV = np.array([0, 120, 130])
FIRE_HIGH_HSV = np.array([35, 255, 255])

# 火情判定阈值
MIN_FIRE_PIXEL_RATIO = 0.015    # 最小火焰像素占比（轻微预警）
DANGER_FIRE_PIXEL_RATIO = 0.06  # 高危火灾像素占比
MOTION_THRESHOLD = 3.5          # 火焰动态分数阈值

# 预警计时缓存
WARNING_HOLD_SEC = 2.0

# =========================================================
# 2.1 华为云IoT设备配置
# =========================================================
IOT_CLIENT_ID = "6a43ba2c7f2e6c302f80931c_SF001_0_0_2026070216"
IOT_USERNAME = "6a43ba2c7f2e6c302f80931c_SF001"
IOT_PASSWORD = "bc4799b499550b8be63c94c0701bc3c8888306e60fdd92fdebdf3da0c7475d2d"
IOT_HOST = "17f1fc05ef.st1.iotda-device.cn-north-4.myhuaweicloud.com"
IOT_PORT = 1883

# 上报主题（华为云IoT固定格式）
IOT_PUB_TOPIC = "$oc/devices/{}/sys/properties/report".format(IOT_CLIENT_ID)
# 下行命令订阅主题（接收云平台下发的数据）
IOT_SUB_COMMAND_TOPIC = "$oc/devices/{}/sys/commands/#".format(IOT_CLIENT_ID)
# 下行消息订阅主题
IOT_SUB_MESSAGE_TOPIC = "$oc/devices/{}/sys/messages/down".format(IOT_CLIENT_ID)
# 属性查询/设置订阅主题
IOT_SUB_PROPERTY_TOPIC = "$oc/devices/{}/sys/properties/set/#".format(IOT_CLIENT_ID)

# 服务类型
SERVICE_TYPE = "smartFire"

# IoT上报间隔（秒）
IOT_REPORT_INTERVAL = 5


# =========================================================
# 3. 目录初始化
# =========================================================
def ensure_dirs():
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.makedirs(SAVE_DIR, exist_ok=True)
    print("=" * 70)
    print("森林火情实时监测预警系统（华为云IoT融合版）")
    print("=" * 70)
    print("项目存储目录：", PROJECT_DIR)
    print("火情截图保存目录：", SAVE_DIR)


# =========================================================
# 4. 中文绘制工具（完全复用原代码）
# =========================================================
def get_chinese_font(font_size=24):
    font_paths = [
        r"C:\Windows\Fonts\msyh.ttc",
        r"C:\Windows\Fonts\simhei.ttf",
        r"C:\Windows\Fonts\simsun.ttc",
    ]
    for font_path in font_paths:
        if os.path.exists(font_path):
            return ImageFont.truetype(font_path, font_size)
    return ImageFont.load_default()


def put_chinese_text(img_bgr, text, position, font_size=24, color=(255, 255, 255)):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil_img)
    font = get_chinese_font(font_size)
    b, g, r = color
    draw.text(position, text, font=font, fill=(r, g, b))
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)


def show_image_in_notebook(img_bgr, title="火情截图"):
    if notebook_display is None:
        print("当前环境不支持Jupyter图片预览")
        return
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    notebook_display(pil_img)


# =========================================================
# 5. 火情监测核心类（融合华为云IoT双向通信）
# =========================================================
class ForestFireMonitor:
    def __init__(self, camera_index=0):
        self.camera_index = camera_index
        self.cap = None
        self.running = False

        # FPS统计
        self.frame_count = 0
        self.fps = 0
        self.fps_times = deque(maxlen=30)

        # 帧缓存（用于计算动态变化）
        self.prev_gray = None
        self.motion_history = deque(maxlen=25)

        # 预警状态缓存
        self.last_warning_time = 0.0
        self.fire_level = "无火情"
        self.fire_level_num = 0    # 火情数字等级：0=无火情, 1=疑似火情, 2=轻度火情, 3=高危火灾
        self.warning_text = "监测正常，未检测到火焰"
        self.fire_pixel_ratio = 0.0
        self.motion_score = 0.0

        # ==================== 华为云IoT相关 ====================
        # 云端下发的传感器数据（由on_message回调更新）
        self.cloud_co2 = 0.0          # 云端下发的CO2浓度（ppm）
        self.cloud_smoke = 0.0        # 云端下发的烟雾浓度
        self.cloud_alert_level = 0    # 云端下发的预警级别（0=正常,1=注意,2=警告,3=危险）

        # MQTT客户端
        self.iot_client = None
        self.iot_connected = False
        self.last_iot_report_time = 0.0

        # 上报计数
        self.iot_report_count = 0

        # 线程锁（保护云端数据读写）
        self.data_lock = threading.Lock()

    # ==================== 华为云IoT连接 ====================
    def connect_iot(self):
        """连接华为云IoT平台"""
        # 注意：paho-mqtt 2.x + VERSION2 回调需要5个参数
        def on_connect(client, userdata, flags, reason_code, properties):
            if reason_code == 0:
                self.iot_connected = True
                print("✅ 华为云IoT连接认证成功！")
                self._subscribe_iot_topics(client)
            else:
                self.iot_connected = False
                print(f"❌ IoT连接失败，错误码：{reason_code}")

        def on_message(client, userdata, msg):
            """接收华为云下发的消息（co2、smoke、alert_level）"""
            try:
                payload_str = msg.payload.decode("utf-8")
                print(f"\n📥 [IoT下行] 收到云端数据 | 主题：{msg.topic}")
                print(f"   原始数据：{payload_str}")

                data = json.loads(payload_str)
                properties = {}

                if "paras" in data:
                    properties = data["paras"]
                elif "services" in data:
                    for svc in data.get("services", []):
                        if "properties" in svc:
                            properties.update(svc["properties"])
                else:
                    properties = data

                with self.data_lock:
                    if "co2" in properties:
                        self.cloud_co2 = float(properties["co2"])
                        print(f"   ✅ 更新 co2 = {self.cloud_co2} ppm")
                    if "smoke" in properties:
                        self.cloud_smoke = float(properties["smoke"])
                        print(f"   ✅ 更新 smoke = {self.cloud_smoke}")
                    if "alert_level" in properties:
                        self.cloud_alert_level = int(properties["alert_level"])
                        print(f"   ✅ 更新 alert_level = {self.cloud_alert_level}")

                print(f"   当前云端数据：co2={self.cloud_co2}, smoke={self.cloud_smoke}, alert_level={self.cloud_alert_level}")

            except json.JSONDecodeError:
                print(f"   ⚠️ 数据解析失败，非JSON格式：{payload_str}")
            except Exception as e:
                print(f"   ⚠️ 处理云端数据异常：{e}")

        def on_disconnect(client, userdata, flags, reason_code, properties):
            self.iot_connected = False
            print(f"🔌 IoT连接断开，reason_code={reason_code}，将自动重连...")

        self.iot_client = mqtt_client.Client(CallbackAPIVersion.VERSION2, IOT_CLIENT_ID)
        self.iot_client.username_pw_set(IOT_USERNAME, IOT_PASSWORD)
        self.iot_client.on_connect = on_connect
        self.iot_client.on_message = on_message
        self.iot_client.on_disconnect = on_disconnect

        try:
            print("🔄 正在连接华为云IoT平台...")
            self.iot_client.connect(IOT_HOST, IOT_PORT, keepalive=60)
            self.iot_client.loop_start()
            time.sleep(1.5)
            return True
        except Exception as e:
            print(f"❌ IoT连接异常：{e}")
            self.iot_connected = False
            return False

    def _subscribe_iot_topics(self, client):
        """订阅华为云IoT下行主题（接收co2、smoke、alert_level）"""
        topics = [
            (IOT_SUB_COMMAND_TOPIC, 1),
            (IOT_SUB_MESSAGE_TOPIC, 1),
            (IOT_SUB_PROPERTY_TOPIC, 1),
        ]
        for topic, qos in topics:
            result = client.subscribe(topic, qos)
            if result[0] == 0:
                print(f"   📡 订阅成功：{topic}")
            else:
                print(f"   ❌ 订阅失败：{topic}")

    def publish_fire_data(self):
        """向华为云上报火情数据（仅fire_type，long类型）"""
        if not self.iot_connected or self.iot_client is None:
            return

        # fire_type: long(整型), 0=无火情, 1=疑似火情, 2=轻度火情, 3=高危火灾
        fire_type = int(self.fire_level_num)

        payload = {
            "services": [
                {
                    "service_id": SERVICE_TYPE,
                    "properties": {
                        "fire_type": fire_type
                    }
                }
            ]
        }

        msg = json.dumps(payload)
        result = self.iot_client.publish(IOT_PUB_TOPIC, msg, qos=1)
        status, mid = result[0], result[1]
        self.iot_report_count += 1

        # ========== 在Jupyter中显示每次发送的值 ==========
        level_labels = {0: "无火情", 1: "疑似火情", 2: "轻度火情", 3: "高危火灾"}
        label = level_labels.get(fire_type, "未知")

        print("-" * 50)
        print(f"📤 第 {self.iot_report_count} 次上报华为云IoT")
        print(f"   fire_type = {fire_type} (long)")
        print(f"   含义：{label}")
        print(f"   MQTT状态码：{status} | msg_id={mid}")
        print(f"   完整报文：{msg}")
        print("-" * 50)

        # Jupyter Notebook 富文本显示
        if display is not None and HTML is not None:
            color_map = {0: "#4CAF50", 1: "#FF9800", 2: "#FF5722", 3: "#F44336"}
            bg_color = color_map.get(fire_type, "#9E9E9E")
            display(HTML(f"""
            <div style="padding:10px; margin:5px 0; border-left:5px solid {bg_color};
                        background:#fafafa; font-family:monospace;">
                <b>📤 第 {self.iot_report_count} 次上报</b><br>
                fire_type = <span style="color:{bg_color}; font-size:1.2em; font-weight:bold;">
                {fire_type}</span> (long) — {label}<br>
                报文: <code>{msg}</code>
            </div>
            """))

    def disconnect_iot(self):
        """断开华为云IoT连接"""
        if self.iot_client is not None:
            self.iot_client.loop_stop()
            self.iot_client.disconnect()
            self.iot_connected = False
            print("🔌 已断开华为云IoT连接")

    # ==================== 摄像头相关（原代码） ====================
    def init_camera(self):
        if self.cap is not None:
            self.cap.release()
        self.cap = cv2.VideoCapture(self.camera_index, cv2.CAP_DSHOW)
        if not self.cap.isOpened():
            self.cap = cv2.VideoCapture(self.camera_index)
        if not self.cap.isOpened():
            print("摄像头打开失败，请检查设备是否占用！")
            return False
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
        self.cap.set(cv2.CAP_PROP_FPS, FPS_TARGET)
        time.sleep(0.5)
        ret, frame = self.cap.read()
        if not ret or frame is None:
            print("摄像头无法读取画面")
            return False
        print("摄像头初始化完成，画面尺寸：", frame.shape)
        return True

    def update_fps(self):
        self.fps_times.append(time.time())
        if len(self.fps_times) >= 2:
            elapsed = self.fps_times[-1] - self.fps_times[0]
            if elapsed > 0:
                self.fps = len(self.fps_times) / elapsed

    def calc_motion_score(self, gray_frame):
        if self.prev_gray is None:
            self.prev_gray = gray_frame.copy()
            return 0.0
        frame_diff = cv2.absdiff(gray_frame, self.prev_gray)
        _, diff_mask = cv2.threshold(frame_diff, 25, 255, cv2.THRESH_BINARY)
        motion_area = np.sum(diff_mask) / 255
        total_area = gray_frame.shape[0] * gray_frame.shape[1]
        score = (motion_area / total_area) * 100
        self.prev_gray = gray_frame.copy()
        self.motion_history.append(score)
        return np.mean(self.motion_history)

    def detect_fire_mask(self, bgr_img):
        hsv = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2HSV)
        fire_mask = cv2.inRange(hsv, FIRE_LOW_HSV, FIRE_HIGH_HSV)
        kernel = np.ones((3, 3), np.uint8)
        fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_OPEN, kernel)
        fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_CLOSE, kernel)
        fire_pixel = np.sum(fire_mask) / 255
        all_pixel = bgr_img.shape[0] * bgr_img.shape[1]
        ratio = fire_pixel / all_pixel
        return fire_mask, ratio

    def judge_fire_level(self, fire_ratio, motion_val):
        """火情分级判定，同时设置文字等级和数字等级"""
        now = time.time()
        if fire_ratio < MIN_FIRE_PIXEL_RATIO or motion_val < MOTION_THRESHOLD:
            self.fire_level = "无火情"
            self.fire_level_num = 0
            self.warning_text = "监测正常，未检测到火焰"
            self.last_warning_time = 0
            return

        if self.last_warning_time == 0:
            self.last_warning_time = now
        hold_time = now - self.last_warning_time

        if hold_time < WARNING_HOLD_SEC:
            self.fire_level = "疑似火情"
            self.fire_level_num = 1
            self.warning_text = "提示：检测到疑似火光，持续观测中"
        elif fire_ratio >= DANGER_FIRE_PIXEL_RATIO:
            self.fire_level = "高危火灾"
            self.fire_level_num = 3
            self.warning_text = "【严重警报】大面积明火，立即处置火情！"
        else:
            self.fire_level = "轻度火情"
            self.fire_level_num = 2
            self.warning_text = "【火情预警】检测到明火区域，请注意防范"

    def draw_fire_contour(self, image, fire_mask):
        contours, _ = cv2.findContours(fire_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 80:
                continue
            x, y, w, h = cv2.boundingRect(cnt)
            if self.fire_level_num == 3:
                color = (0, 0, 255)
            else:
                color = (0, 220, 255)
            cv2.rectangle(image, (x, y), (x + w, y + h), color, 2)
        return image

    def add_info_panel(self, image):
        h, w = image.shape[:2]
        cv2.rectangle(image, (0, 0), (w, 170), (0, 0, 0), -1)
        cv2.rectangle(image, (0, h - 45), (w, h), (0, 0, 0), -1)

        image = put_chinese_text(image, "森林火情实时监测系统（IoT联网）", (15, 8), 24, (0, 255, 0))
        image = put_chinese_text(image, f"火情等级：{self.fire_level} (Lv.{self.fire_level_num})",
                                (15, 40), 22, (255, 255, 255))

        if self.fire_level_num == 3:
            warn_color = (0, 0, 255)
        elif self.fire_level_num == 2 or self.fire_level_num == 1:
            warn_color = (0, 220, 255)
        else:
            warn_color = (200, 200, 200)
        image = put_chinese_text(image, f"系统提示：{self.warning_text}", (15, 70), 22, warn_color)
        image = put_chinese_text(image, f"FPS:{self.fps:.1f} 火焰占比:{self.fire_pixel_ratio:.3f} 动态分数:{self.motion_score:.1f}",
                                (15, 100), 20, (255, 255, 255))

        with self.data_lock:
            cloud_info = f"云端数据 | CO2:{self.cloud_co2:.0f}ppm Smoke:{self.cloud_smoke:.1f} Alert:{self.cloud_alert_level}"
        iot_status = "IoT已连接✅" if self.iot_connected else "IoT未连接❌"
        image = put_chinese_text(image, f"{cloud_info}  {iot_status}",
                                (15, 130), 18, (180, 255, 180))

        image = put_chinese_text(image, "Q退出程序 | S保存火情截图", (15, h - 38), 21, (220, 220, 220))
        return image

    def save_frame(self, frame):
        save_name = f"fire_alarm_{self.fire_level}_{time.strftime('%Y%m%d_%H%M%S')}.jpg"
        save_path = os.path.join(SAVE_DIR, save_name)
        preview = frame.copy()
        preview = put_chinese_text(preview, "火情截图已保存", (20, 150), 30, (0, 255, 255))
        cv2.imencode(".jpg", preview)[1].tofile(save_path)
        saved_data = np.fromfile(save_path, dtype=np.uint8)
        saved_img = cv2.imdecode(saved_data, cv2.IMREAD_COLOR)
        if saved_img is not None:
            cv2.imshow("火情截图预览", saved_img)
            show_image_in_notebook(saved_img, title=f"截图存储路径：{save_path}")
        print("火情截图保存完成：", save_path)

    def process_frame(self, raw_frame):
        img = cv2.flip(raw_frame, 1)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        fire_mask, self.fire_pixel_ratio = self.detect_fire_mask(img)
        self.motion_score = self.calc_motion_score(gray)
        self.judge_fire_level(self.fire_pixel_ratio, self.motion_score)
        img = self.draw_fire_contour(img, fire_mask)
        img = self.add_info_panel(img)
        self.frame_count += 1
        self.update_fps()

        # ========== 定时向华为云上报火情数据 ==========
        now = time.time()
        if now - self.last_iot_report_time >= IOT_REPORT_INTERVAL:
            self.publish_fire_data()
            self.last_iot_report_time = now

        return img

    def run(self):
        if not self.init_camera():
            return

        self.connect_iot()

        self.running = True
        print("\n" + "=" * 70)
        print("森林火情监测系统启动成功（华为云IoT融合版）")
        print("识别原理：火焰HSV色彩阈值 + 帧间动态变化检测")
        print("IoT上报：fire_type（long, 0=无火情 1=疑似 2=轻度 3=高危）")
        print("IoT接收：co2（CO2浓度）、smoke（烟雾）、alert_level（预警级别）")
        print("按键操作：Q退出 | S保存当前火情截图")
        print("=" * 70 + "\n")

        proc_img = None
        try:
            while self.running:
                ret, frame = self.cap.read()
                if not ret or frame is None:
                    print("摄像头画面中断，停止监测")
                    break
                proc_img = self.process_frame(frame)
                cv2.imshow("森林火情监测系统", proc_img)
                key = cv2.waitKey(1) & 0xFF
                if key == ord("q"):
                    print("用户按下Q，结束监测程序")
                    break
                elif key == ord("s") and proc_img is not None:
                    self.save_frame(proc_img)
        except KeyboardInterrupt:
            print("\n程序被手动中断")
        except Exception as err:
            print("监测运行异常：", err)
            import traceback
            traceback.print_exc()
        finally:
            self.stop()

    def stop(self):
        self.running = False
        self.disconnect_iot()
        if self.cap is not None:
            self.cap.release()
        cv2.destroyAllWindows()
        print("\n摄像头资源释放完成，程序退出")
        print("本次监测总处理帧数：", self.frame_count)


# =========================================================
# 6. 程序入口
# =========================================================
def main():
    ensure_dirs()
    monitor = ForestFireMonitor(camera_index=CAMERA_INDEX)
    monitor.run()


if __name__ == "__main__":
    main()

In [ ]:
# =========================================================
# 【诊断工具 v2】对比测试 — 精准定位华为云收不到 fire_type 的原因
# service_id = smartFire
# =========================================================
import json
import time
from paho.mqtt import client as mqtt_client
from paho.mqtt.client import CallbackAPIVersion

try:
    from IPython.display import display, HTML
except Exception:
    display = None
    HTML = None

# 设备配置
IOT_CLIENT_ID = "6a43ba2c7f2e6c302f80931c_SF001_0_0_2026070216"
IOT_USERNAME = "6a43ba2c7f2e6c302f80931c_SF001"
IOT_PASSWORD = "bc4799b499550b8be63c94c0701bc3c8888306e60fdd92fdebdf3da0c7475d2d"
IOT_HOST = "17f1fc05ef.st1.iotda-device.cn-north-4.myhuaweicloud.com"
IOT_PORT = 1883

PUB_TOPIC = "$oc/devices/{}/sys/properties/report".format(IOT_CLIENT_ID)
SERVICE_TYPE = "smartFire"

test_client = None
test_connected = False
send_count = 0

def on_connect(client, userdata, flags, reason_code, properties):
    global test_connected
    if reason_code == 0:
        test_connected = True
        print("✅ MQTT连接成功！")
    else:
        test_connected = False
        print(f"❌ 连接失败，reason_code={reason_code}")

def on_publish(client, userdata, mid, reason_code=None, properties=None):
    print(f"   📨 Broker确认 msg_id={mid}")

def on_disconnect(client, userdata, flags, reason_code, properties):
    global test_connected
    test_connected = False
    print(f"🔌 断开，reason_code={reason_code}")

def send_test(name, service_id, properties_dict):
    global test_client, send_count
    payload = {
        "services": [{
            "service_id": service_id,
            "properties": properties_dict
        }]
    }
    msg = json.dumps(payload, ensure_ascii=False)
    result = test_client.publish(PUB_TOPIC, msg, qos=1)
    status, mid = result[0], result[1]
    send_count += 1

    print(f"\n📤 测试{send_count}：{name}")
    print(f"   service_id={service_id}")
    print(f"   properties={properties_dict}")
    print(f"   报文: {msg}")
    print(f"   MQTT状态: {status} | msg_id={mid}")

    if display and HTML:
        display(HTML(f"""
        <div style="padding:8px; margin:4px 0; border-left:4px solid #2196F3;
                    background:#fafafa; font-family:monospace; font-size:13px;">
            <b>📤 {name}</b> | service={service_id}<br>
            <code>{msg}</code>
        </div>
        """))

def run_diagnostic():
    global test_client, send_count

    print("=" * 60)
    print("华为云IoT 诊断工具 v2 — service_id = smartFire")
    print("=" * 60)

    test_client = mqtt_client.Client(CallbackAPIVersion.VERSION2, IOT_CLIENT_ID)
    test_client.username_pw_set(IOT_USERNAME, IOT_PASSWORD)
    test_client.on_connect = on_connect
    test_client.on_publish = on_publish
    test_client.on_disconnect = on_disconnect

    try:
        test_client.connect(IOT_HOST, IOT_PORT, keepalive=60)
        test_client.loop_start()
        time.sleep(2)
    except Exception as e:
        print(f"❌ 连接异常：{e}")
        return

    if not test_connected:
        print("❌ 连接失败，请检查凭证")
        test_client.loop_stop()
        return

    print("\n" + "=" * 60)
    print("依次发送 fire_type 0→1→2→3（service=smartFire）")
    print("=" * 60)

    for level in [0, 1, 2, 3]:
        level_labels = {0: "无火情", 1: "疑似火情", 2: "轻度火情", 3: "高危火灾"}
        send_test(
            f"fire_type={level} ({level_labels[level]})",
            SERVICE_TYPE,
            {"fire_type": level}
        )
        time.sleep(3)

    print("\n" + "=" * 60)
    print(f"发送完成！共 {send_count} 次，service_id = {SERVICE_TYPE}")
    print("请在华为云控制台确认是否收到 fire_type 数据")
    print("=" * 60)

    time.sleep(5)
    test_client.loop_stop()
    test_client.disconnect()
    print("结束。")

run_diagnostic()